# Model 03 — K-Nearest Neighbors (Motor Incipient Failure Prediction)

An electric motor does not fail suddenly. Long before a catastrophic breakdown, it sends signals: vibration levels creep upward, bearing temperatures climb, operating current increases, and the dominant frequency in the spectrum begins to drift. These changes are subtle, overlapping, and non-linear — which is precisely what makes this problem interesting from a modeling perspective, and what makes it expensive to miss in production.

This project applies **K-Nearest Neighbors** to classify motor operating states as normal or incipient failure based on six condition-monitoring signals. The choice of model reflects the nature of the problem: incipient failures do not follow a linear rule like "if vibration > threshold, then alert". They emerge from specific *combinations* of signals — an operating region in the feature space where the motor is physically degraded. KNN detects these regions directly, by asking: *"what do the motors most similar to this one look like?"* No assumptions about distribution shape or linearity required.

The dataset simulates a realistic condition monitoring scenario with two well-separated operating regimes — normal operation (n=1,000) and incipient failure (n=500) — with feature distributions grounded in real motor physics. The large separation between classes, confirmed by permutation importance and the 2D decision map, makes this a clean demonstration of why local, instance-based methods outperform global linear models for anomaly detection in machinery.

---
**Data source:** `motor_failure_prediction_data.csv`  
**Output:** metrics, permutation importance, 2D decision boundary map, McNemar comparison vs Logistic Regression, and a motor condition simulator


## Section 1 — Setup

Reproducibility matters in predictive maintenance: when a model is retrained on updated data, we need the result to be deterministic for auditing purposes. We fix a global seed, import all dependencies upfront, and configure plot aesthetics once.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Libraries loaded. Random state:", RANDOM_STATE)


## Section 2 — Load Data

The dataset contains **1,500 motor operating records** from a condition monitoring system. Each row represents one measurement window, described by six physical signals captured from the motor: vibration RMS and peak-to-peak (mm/s), bearing temperature (°C), operating current (A), dominant frequency in the vibration spectrum (Hz), and load percentage. The target `incipient_failure` is binary — 0 for normal operation, 1 for incipient failure state.

> *Note: The dataset was integrated using Maintenance datasets that models two distinct motor operating regimes — normal and incipient failure — based on real sensor distributions from electric motor condition monitoring. Normal motors: vib RMS ~2.2 mm/s, bearing temp ~60°C, current ~28A. Incipient failure motors: vib RMS ~4.2 mm/s, bearing temp ~82°C, current ~35A. The separation between regimes is physically grounded: these are the signal shifts that precede mechanical failure in induction motors.*

The try/except pattern makes the notebook portable between local and Google Colab environments.


In [ ]:
try:
    df = pd.read_csv("motor_failure_prediction_data.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/LozanoLsa/Motor_Failures_Prediction/main/motor_failure_prediction_data.csv")
    # FileNotFoundError is intentionally specific — a bare except would silently swallow real data errors

FEATURES = ["vib_rms_mms", "vib_peak_to_peak_mms", "bearing_temp_c",
             "motor_current_a", "dominant_freq_hz", "load_pct"]
TARGET = "incipient_failure"

df.head()


## Section 3 — Quick Sanity Checks

We validate the loaded data before touching the model. Six numeric features, no categoricals to encode, no scaling leakage to worry about at this stage. The class ratio is 2:1 (normal vs failure) — intentionally imbalanced to reflect a realistic monitoring scenario where failures are a minority event. Recall on the failure class is therefore the key metric, not overall accuracy.


In [ ]:
# Sanity checks. Real datasets usually try to hurt you 🙂
print("Shape:", df.shape)
print("\n--- Data types ---")
df.info()


In [ ]:
print("--- Missing values ---")
print(df.isna().sum())
print("\n--- Target class balance ---")
print(df[TARGET].value_counts())
print("\n--- Failure rate ---")
print(df[TARGET].value_counts(normalize=True).rename({0: "normal", 1: "incipient_failure"}))


In [ ]:
print("--- Numeric summary ---")
df.describe().round(2)


## Section 4 — Exploratory Data Analysis

Before modeling, we want to confirm three things: (1) that the two classes occupy distinct regions in the feature space — which would justify a distance-based model; (2) which features show the strongest separation between normal and failure states; and (3) whether the class separation is clean enough to warrant confidence in the predictions.

The EDA here is not decorative — it is the physical validation that the sensor signals are actually carrying the information we expect from motor physics.


In [ ]:
# Overlapping histograms — where do the class distributions separate?
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    axes[i].hist(df[df[TARGET]==0][col], bins=30, alpha=0.6,
                 color="#4C9BE8", edgecolor="white", label="Normal")
    axes[i].hist(df[df[TARGET]==1][col], bins=30, alpha=0.6,
                 color="#E8574C", edgecolor="white", label="Incipient Failure")
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].legend(fontsize=8)
plt.suptitle("Sensor Signal Distributions by Motor State",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Boxplots by class — median shift and spread per feature
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(FEATURES):
    for val, color in [(0, "#4C9BE8"), (1, "#E8574C")]:
        subset = df[df[TARGET] == val][col]
        axes[i].boxplot([subset.values], positions=[val],
                        patch_artist=True,
                        boxprops=dict(facecolor=color, alpha=0.7),
                        medianprops=dict(color="white", linewidth=2))
    axes[i].set_title(col.replace("_", " ").title())
    axes[i].set_xticks([0, 1])
    axes[i].set_xticklabels(["Normal", "Failure"])
plt.suptitle("Feature Distribution by Motor State (Blue = Normal, Red = Failure)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plot — vib_rms vs bearing_temp (the two most physically informative signals)
# If the two classes cluster cleanly here, KNN will perform well
fig, ax = plt.subplots(figsize=(8, 6))
scatter_colors = {0: "#4C9BE8", 1: "#E8574C"}
scatter_labels = {0: "Normal", 1: "Incipient Failure"}
for cls in [0, 1]:
    mask = df[TARGET] == cls
    ax.scatter(df.loc[mask, "vib_rms_mms"], df.loc[mask, "bearing_temp_c"],
               alpha=0.4, s=20, color=scatter_colors[cls], label=scatter_labels[cls])
ax.set_xlabel("Vibration RMS (mm/s)")
ax.set_ylabel("Bearing Temperature (°C)")
ax.set_title("Vibration RMS vs Bearing Temperature — Motor State", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()
print("Two clearly separated clusters — strong signal for a distance-based model.")


In [ ]:
# Correlation matrix
fig, ax = plt.subplots(figsize=(8, 6))
corr = df[FEATURES + [TARGET]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="RdBu_r",
            vmin=-1, vmax=1, linewidths=0.5, ax=ax, annot_kws={"size": 9})
ax.set_title("Feature Correlation Matrix", fontweight="bold")
plt.tight_layout()
plt.show()


## Section 5 — Preprocessing

KNN makes decisions entirely based on distance — it finds the *k* most similar motors in the training set and votes on the outcome. This means the scale of each feature directly affects which neighbors are found. A variable measured in Amperes (range ~25–40) and one measured in °C (range ~50–95) would produce distances dominated by temperature, simply because its absolute values are larger. **StandardScaler** removes this distortion by centering each feature to zero mean and unit variance before computing distances.

All preprocessing is wrapped inside a **sklearn Pipeline** alongside the classifier. This ensures that hyperparameter tuning via GridSearchCV operates correctly — the scaler is fit only on the training fold in each cross-validation split, never on the validation fold.


In [ ]:
X = df[FEATURES]
y = df[TARGET]

# Stratified split — preserves 2:1 class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE
)

# Pipeline: StandardScaler feeds directly into KNN
# Scaling inside the pipeline prevents data leakage during cross-validation
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("knn",    KNeighborsClassifier())
])

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"Failure rate : train={y_train.mean():.3f}  test={y_test.mean():.3f}")


## Section 6 — Train Model (GridSearchCV)

KNN has three key hyperparameters that require tuning: the number of neighbors *k*, the distance metric (p=1 Manhattan, p=2 Euclidean), and the weighting scheme (uniform or distance-weighted). We search over a grid of combinations using 5-fold cross-validation, so the optimal configuration is selected based on held-out performance rather than training performance.

Distance-weighted KNN (`weights='distance'`) gives closer neighbors more influence — physically meaningful for motor anomalies, where the most similar historical observations should carry more diagnostic weight than distant ones.


In [ ]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights":     ["uniform", "distance"],
    "knn__p":           [1, 2]   # 1=Manhattan, 2=Euclidean
}

grid = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best hyperparameters:", grid.best_params_)
print(f"Best CV accuracy    : {grid.best_score_:.4f}")

best_knn = grid.best_estimator_


## Section 7 — Evaluate

In predictive maintenance, the asymmetry of error costs is extreme: a **false negative** (predicting normal when the motor is actually failing) means a missed intervention — the motor continues running toward catastrophic failure. A **false positive** (flagging a healthy motor) means an unnecessary inspection — expensive but manageable. This asymmetry means we weight Recall for the failure class above Accuracy as the primary performance signal.

The dataset's clean class separation — physically grounded in the large difference between normal and failure-state sensor distributions — produces a model with very high performance across all metrics, which the probability distribution plot confirms.


In [ ]:
y_pred = best_knn.predict(X_test)
y_prob = best_knn.predict_proba(X_test)[:, 1]

acc  = accuracy_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print(f"Accuracy  : {acc:.4f}")
print(f"AUC-ROC   : {auc:.4f}")
print(f"Precision (Failure): {prec:.4f}")
print(f"Recall    (Failure): {rec:.4f}")
print(f"F1        (Failure): {f1:.4f}")
print("\n", classification_report(y_test, y_pred, target_names=["Normal", "Incipient Failure"]))


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
labels = ["Normal", "Incipient Failure"]

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.set_title("Confusion Matrix", fontweight="bold")
plt.tight_layout()
plt.show()
print("Rows = what actually happened. Columns = what the model predicted. Diagonal = correct predictions.")


In [ ]:
# ROC + Precision-Recall curves
fpr, tpr, _ = roc_curve(y_test, y_prob)
prec_c, rec_c, _ = precision_recall_curve(y_test, y_prob)
ap = average_precision_score(y_test, y_prob)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color="#4C9BE8", lw=2, label=f"AUC = {auc:.3f}")
axes[0].plot([0,1],[0,1], linestyle="--", color="gray", lw=1, label="Random baseline")
axes[0].fill_between(fpr, tpr, alpha=0.1, color="#4C9BE8")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].legend()

axes[1].plot(rec_c, prec_c, color="#E8574C", lw=2, label=f"AP = {ap:.3f}")
axes[1].axhline(y_test.mean(), linestyle="--", color="gray", lw=1,
                label=f"Baseline (failure rate = {y_test.mean():.2f})")
axes[1].fill_between(rec_c, prec_c, alpha=0.1, color="#E8574C")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Predicted probability distribution by true class
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(y_prob[y_test == 0], bins=30, alpha=0.6, color="#4C9BE8",
        label="Actual: Normal", edgecolor="white")
ax.hist(y_prob[y_test == 1], bins=30, alpha=0.6, color="#E8574C",
        label="Actual: Incipient Failure", edgecolor="white")
ax.axvline(0.5, color="black", linestyle="--", lw=1.5, label="Decision threshold (0.5)")
ax.set_xlabel("Predicted Failure Probability")
ax.set_ylabel("Count")
ax.set_title("Predicted Probability Distribution by True Class", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


## Section 8 — Interpretability: Permutation Importance + 2D Decision Map

KNN has no native coefficients — it does not learn a parameterized decision boundary. Interpretability therefore comes from two complementary tools:

**Permutation Importance** answers: *"if I shuffle this feature across all test samples, how much does accuracy drop?"* A large drop means the model was genuinely relying on that feature — not just coincidentally correlated. This is model-agnostic and works regardless of the underlying algorithm.

**The 2D decision boundary map** answers: *"where in the feature space does the model switch from Normal to Failure?"* Plotted on the two strongest features, it reveals the geometric shape of the KNN decision regions — and confirms whether the model has learned a physically plausible boundary or something that looks like overfitting.


In [ ]:
# Permutation importance — computed on the test set
# Shuffles each feature 10 times and measures the average accuracy drop
result = permutation_importance(
    best_knn, X_test, y_test,
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

imp_df = pd.DataFrame({
    "Feature": FEATURES,
    "Importance Mean": result.importances_mean,
    "Importance Std":  result.importances_std
}).sort_values("Importance Mean", ascending=False).reset_index(drop=True)

print("Permutation Importance (accuracy drop when feature is shuffled):")
print(imp_df.round(5).to_string(index=False))


In [ ]:
# Permutation importance bar chart with error bars
imp_sorted = imp_df.sort_values("Importance Mean", ascending=True)
colors_imp = ["#E8574C" if v > 0 else "#888" for v in imp_sorted["Importance Mean"]]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(range(len(imp_sorted)), imp_sorted["Importance Mean"],
        xerr=imp_sorted["Importance Std"],
        color=colors_imp, edgecolor="white", height=0.6,
        error_kw=dict(ecolor="white", capsize=4, linewidth=1.5))
ax.set_yticks(range(len(imp_sorted)))
ax.set_yticklabels([f.replace("_", " ").title() for f in imp_sorted["Feature"]])
ax.axvline(0, color="white", linewidth=1)
ax.set_xlabel("Mean Accuracy Drop (feature shuffled)")
ax.set_title("Permutation Importance — KNN Motor Failure Model", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 2D decision boundary map — vib_rms_mms vs bearing_temp_c
# All other features held at their training-set mean
# This reveals the shape of KNN's decision regions in the 2 most informative dimensions

scaler_fit = best_knn.named_steps["scaler"]
knn_fit    = best_knn.named_steps["knn"]

x_feat, y_feat = "vib_rms_mms", "bearing_temp_c"
x_idx = FEATURES.index(x_feat)
y_idx = FEATURES.index(y_feat)

x_vals = np.linspace(df[x_feat].min() - 0.5, df[x_feat].max() + 0.5, 200)
y_vals = np.linspace(df[y_feat].min() - 2,   df[y_feat].max() + 2,   200)
xx, yy = np.meshgrid(x_vals, y_vals)

# Build grid with all features at mean
X_mean = scaler_fit.mean_
grid_mat = np.tile(X_mean, (xx.ravel().shape[0], 1))
grid_mat[:, x_idx] = xx.ravel()
grid_mat[:, y_idx] = yy.ravel()

# Scale and predict
grid_scaled = scaler_fit.transform(grid_mat)
probs = knn_fit.predict_proba(grid_scaled)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
cont = ax.contourf(xx, yy, probs, levels=20, cmap="RdYlGn_r", alpha=0.85)
plt.colorbar(cont, ax=ax, label="P(Incipient Failure)")

for cls, color, label in [(0, "#4C9BE8", "Normal"), (1, "#E8574C", "Incipient Failure")]:
    mask = df[TARGET] == cls
    ax.scatter(df.loc[mask, x_feat], df.loc[mask, y_feat],
               s=14, alpha=0.45, color=color, edgecolors="none", label=label)

ax.set_xlabel("Vibration RMS (mm/s)")
ax.set_ylabel("Bearing Temperature (°C)")
ax.set_title("KNN Decision Boundary Map — Vibration RMS vs Bearing Temperature\n"
             "(all other features held at training mean)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


### Summary of Practical Signals

The permutation importance and decision map tell a physically coherent story:

- **`vib_rms_mms` is the top driver** — vibration RMS is the primary discriminating signal in rotating machinery diagnostics. A shift from ~2.2 to ~4.2 mm/s represents a near-doubling of vibration energy, which is well above typical alarm thresholds in ISO 10816 standards.
- **`vib_peak_to_peak_mms` and `bearing_temp_c` are close behind** — peak-to-peak captures impact events (looseness, imbalance) while bearing temperature reflects friction buildup. Together, these three signals form the core diagnostic triad.
- **`dominant_freq_hz` carries meaningful signal** — a drop in dominant frequency is consistent with developing misalignment, which shifts the spectral peak.
- **`motor_current_a` and `load_pct` contribute less** — current and load are important contextual variables, but in this dataset their predictive contribution is smaller once vibration and temperature are accounted for.
- **The decision boundary is nonlinear and sharp** — the 2D map shows a clean, non-straight frontier between the two operating regimes. A linear model could not replicate this boundary without feature engineering. KNN captures it naturally.


## Section 9 — Statistical Validation

Two validation questions are worth answering explicitly: (1) are the results stable across different data partitions? and (2) is KNN genuinely better than a simpler linear model — or is the performance difference within sampling noise? We answer both with cross-validation and a McNemar test.


In [ ]:
# 5-fold stratified cross-validation on training set
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_acc = cross_val_score(best_knn, X_train, y_train, cv=cv, scoring="accuracy")
cv_auc = cross_val_score(best_knn, X_train, y_train, cv=cv, scoring="roc_auc")
cv_f1  = cross_val_score(best_knn, X_train, y_train, cv=cv, scoring="f1")

print("5-Fold Stratified Cross-Validation (training set only):")
print(f"  Accuracy : {cv_acc.mean():.4f} ± {cv_acc.std():.4f}")
print(f"  AUC-ROC  : {cv_auc.mean():.4f} ± {cv_auc.std():.4f}")
print(f"  F1       : {cv_f1.mean():.4f}  ± {cv_f1.std():.4f}")


In [ ]:
# McNemar test — KNN vs Logistic Regression
# Tests whether the disagreement between classifiers is statistically significant
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)

# Disagreement counts
# n01: KNN wrong, LR right | n10: KNN right, LR wrong
n01 = int(np.sum((y_pred != y_test.values) & (y_pred_lr == y_test.values)))
n10 = int(np.sum((y_pred == y_test.values) & (y_pred_lr != y_test.values)))

print("McNemar test: KNN vs Logistic Regression")
print(f"  Cases KNN wrong, LR right (n01) : {n01}")
print(f"  Cases KNN right, LR wrong (n10) : {n10}")

if n01 + n10 > 0:
    stat = (abs(n01 - n10) - 1)**2 / (n01 + n10)
    pval = 1 - chi2.cdf(stat, df=1)
    print(f"  Chi-squared statistic           : {stat:.4f}")
    print(f"  p-value                         : {pval:.4f}")
    if pval < 0.05:
        print("  Conclusion: Difference is statistically significant (p < 0.05)")
    else:
        print("  Conclusion: Difference is NOT statistically significant (p >= 0.05)")
else:
    print()
    print("  Both models made identical predictions on this test set.")
    print("  This is expected when both classifiers reach perfect accuracy on")
    print("  a well-separated dataset — the McNemar test requires at least one")
    print("  disagreement to compute a meaningful statistic.")
    print("  Conclusion: Both KNN and Logistic Regression achieve perfect separation")
    print("  on this dataset. KNN's advantage would become apparent on data with")
    print("  more overlap between classes — its non-linear boundary is not needed here.")

# LR metrics for comparison
print(f"\nLogistic Regression accuracy (test): {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"KNN accuracy (test)                : {accuracy_score(y_test, y_pred):.4f}")


## Section 10 — Motor Condition Simulator

A simple simulator turns the model into a real-time condition monitoring tool. Instead of only scoring historical records, a maintenance engineer can input the current sensor readings for a specific motor and receive an estimated failure probability. This is the most operationally valuable output: not a batch classification, but an interactive risk score that enables conditional decisions — *"this motor is at 23% probability, continue monitoring"* vs *"this motor is at 91%, schedule inspection before end of shift"*.


In [ ]:
def simulate_motor(
    vib_rms_mms: float = 2.2,
    vib_peak_to_peak_mms: float = 5.0,
    bearing_temp_c: float = 60.0,
    motor_current_a: float = 28.0,
    dominant_freq_hz: float = 30.0,
    load_pct: float = 70.0,
    threshold: float = 0.5
) -> tuple:
    """
    Estimate incipient failure probability for a motor with given sensor readings.

    Parameters
    ----------
    vib_rms_mms          : Vibration RMS in mm/s (normal ~2.2, failure ~4.2)
    vib_peak_to_peak_mms : Vibration peak-to-peak mm/s (normal ~5.0, failure ~9.0)
    bearing_temp_c       : Bearing temperature °C (normal ~60, failure ~82)
    motor_current_a      : Motor current Amperes (normal ~28, failure ~35)
    dominant_freq_hz     : Dominant vibration frequency Hz (normal ~30, failure ~25)
    load_pct             : Load percentage (normal ~70%, failure ~85%)
    threshold            : Classification cutoff (default 0.5)
    """
    df_new = pd.DataFrame([{
        "vib_rms_mms":          vib_rms_mms,
        "vib_peak_to_peak_mms": vib_peak_to_peak_mms,
        "bearing_temp_c":       bearing_temp_c,
        "motor_current_a":      motor_current_a,
        "dominant_freq_hz":     dominant_freq_hz,
        "load_pct":             load_pct
    }])

    # Pipeline handles scaling internally
    prob  = best_knn.predict_proba(df_new)[0, 1]
    clase = int(prob >= threshold)

    print("=" * 56)
    print("  MOTOR INCIPIENT FAILURE PROBABILITY SIMULATOR")
    print("=" * 56)
    print(f"  Vibration RMS      : {vib_rms_mms:.2f} mm/s")
    print(f"  Vibration P-P      : {vib_peak_to_peak_mms:.2f} mm/s")
    print(f"  Bearing Temp       : {bearing_temp_c:.1f} °C")
    print(f"  Motor Current      : {motor_current_a:.1f} A")
    print(f"  Dominant Freq      : {dominant_freq_hz:.1f} Hz")
    print(f"  Load               : {load_pct:.1f}%")
    print("-" * 56)
    print(f"  Failure probability: {prob:.3f} ({prob*100:.1f}%)")
    print(f"  Decision threshold : {threshold:.2f}")
    if clase == 1:
        print("  Prediction: >>> INCIPIENT FAILURE — SCHEDULE INSPECTION <<<")
    else:
        print("  Prediction: >>> NORMAL OPERATION — CONTINUE MONITORING <<<")
    print("=" * 56)
    return prob, clase


### Scenario A — Healthy motor within normal operating ranges


In [ ]:
simulate_motor(
    vib_rms_mms=2.0,
    vib_peak_to_peak_mms=5.0,
    bearing_temp_c=60.0,
    motor_current_a=28.0,
    dominant_freq_hz=30.0,
    load_pct=70.0,
    threshold=0.5
)


### Scenario B — Motor showing incipient failure signals


In [ ]:
simulate_motor(
    vib_rms_mms=4.8,
    vib_peak_to_peak_mms=9.8,
    bearing_temp_c=88.0,   # elevated bearing temperature
    motor_current_a=37.0,  # overcurrent — mechanical resistance
    dominant_freq_hz=23.0, # frequency drop — possible misalignment
    load_pct=92.0,         # near-limit load
    threshold=0.5
)


### Scenario C — Borderline motor: vibration elevated but temperature controlled


In [ ]:
# Useful for understanding where the model draws the boundary
simulate_motor(
    vib_rms_mms=3.2,       # elevated but below typical failure zone
    vib_peak_to_peak_mms=6.5,
    bearing_temp_c=68.0,   # slightly elevated
    motor_current_a=30.0,
    dominant_freq_hz=28.5,
    load_pct=78.0,
    threshold=0.5
)


## Final Reflection

This project is not about predicting motor failures with certainty.  
It is about **making incipient failure visible before the breakdown occurs**.

Electric motors do not fail cleanly. They degrade through specific combinations of physical conditions — vibration patterns that worsen gradually, bearings that run progressively hotter, current draw that increases as mechanical resistance builds. These patterns exist in sensor data long before a motor trips a protection relay or stops running. The question is not *whether* the warning signals are there — they almost always are. The question is whether the monitoring system is sensitive enough to catch them.

What this model offers is not a verdict, but a **conversation**:
- *How many degrees of bearing temperature increase move this motor from safe to alert?*
- *At what vibration RMS level should we schedule an unplanned inspection?*
- *Which combination of signals constitutes the boundary between monitoring and acting?*

These questions matter more than the AUC score.

---

*LozanoLsa  |  Operational Excellence · MBB · Machine Learning  |  GitHub: LozanoLsa*
